# Benchmark Plan: Assign Labels (Update 1000 gt_labels in 800M-row v2_patch table)

## Context

From `db_technical_design.md` §2.2 / §2.6:
> *Each time the user updates ground truth labels for patches (e.g., via the patch gallery):*
> *Upsert the `gt_label` in the patch table for specific `patch_id`s as needed.*

This benchmark measures the wall-clock time to update `gt_label` for 1,000
randomly-selected patches in the existing **~800M-row** `v2_patch` table using a
single bulk `UPDATE … WHERE id = ANY(%s)` statement.

## Plan

- **Operation measured**: Single `UPDATE v2_patch SET gt_label = %s, event_ts = NOW() WHERE id = ANY(%s)`
  targeting 1,000 randomly-sampled patch IDs — mirrors a user batch-labelling action.
- **Table size**: ~800,000,200 rows in `v2_patch` (the design target of ≥1B is approached;
  this is the actual table available in the benchmark environment).
- **Index**: `PRIMARY KEY` B-tree on `id` — the WHERE clause uses `= ANY(%s)` which allows
  PG to use index scans for each matched ID.
- **Data distribution**: IDs sampled uniformly at random across the full 1–800M range;
  represents a worst-case scatter-update pattern (no clustering).
- **Environment setup**: Uses the **existing** `v2_patch` table. No data is inserted or
  deleted. Original `gt_label` values are saved before the timed block and restored after.
- **Timing method**: `time.perf_counter()` wraps only the single UPDATE statement;
  connection setup, ID sampling, save/restore, and warm-up are excluded from the timed section.
- **Edge cases**:
  - IDs are sampled uniformly at random (no monotonic cluster) to stress random page access.
  - A warm-up SELECT is performed on the sampled IDs before timing to prime OS/PG caches.
  - The new `gt_label` value (99) is intentionally different from any existing label (0–5).
  - Restore is run inside a `finally` block to guarantee cleanup even on error.
  - Three trials are run to report variance.

## Setup

See `setup_assign_labels_v2_patch.ipynb` for schema details and pre-flight verification.

In [ ]:
import os
import time
import random
import psycopg2
import psycopg2.extras

# ---------------------------------------------------------------------------
# Connection parameters
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TABLE      = 'v2_patch'
BATCH_SIZE = 1_000
NEW_LABEL  = 99          # Sentinel value — not in existing distribution (0-5)
SEED       = 42

conn = psycopg2.connect(DSN)
conn.autocommit = False
cur  = conn.cursor()

# Report version
cur.execute('SELECT version();')
print('Connected to', cur.fetchone()[0].split(',')[0])

# ---------------------------------------------------------------------------
# Get ID range (excluded from timed section)
# ---------------------------------------------------------------------------
cur.execute(f'SELECT MIN(id), MAX(id), COUNT(*) FROM {TABLE};')
min_id, max_id, n_rows = cur.fetchone()
print(f'ID range: {min_id:,} – {max_id:,}  |  Table row count: {n_rows:,}')

# ---------------------------------------------------------------------------
# Sample 1000 random IDs (excluded from timed section)
# ---------------------------------------------------------------------------
rng = random.Random(SEED)
patch_ids = rng.sample(range(min_id, max_id + 1), BATCH_SIZE)
print(f'Sampled {len(patch_ids)} random patch IDs (seed={SEED})')

# ---------------------------------------------------------------------------
# Save original gt_label values (excluded from timed section)
# ---------------------------------------------------------------------------
cur.execute(
    f'SELECT id, gt_label FROM {TABLE} WHERE id = ANY(%s)',
    (patch_ids,)
)
originals = cur.fetchall()          # list of (id, gt_label)
original_map = {row[0]: row[1] for row in originals}
print(f'Saved original gt_label for {len(original_map):,} rows')
assert len(original_map) == BATCH_SIZE, (
    f'Expected {BATCH_SIZE} rows, found {len(original_map)} — some IDs may be missing'
)

# ---------------------------------------------------------------------------
# Warm-up: read the target rows to prime OS/PG caches (NOT timed)
# ---------------------------------------------------------------------------
print('\n--- Warm-up SELECT (not timed) ---')
cur.execute(
    f'SELECT id, gt_label FROM {TABLE} WHERE id = ANY(%s)',
    (patch_ids,)
)
_ = cur.fetchall()
conn.rollback()   # no-op: no writes yet, but keeps txn clean

# ---------------------------------------------------------------------------
# TIMED SECTION: single bulk UPDATE for 1000 patches
# ---------------------------------------------------------------------------
print('\n=== TIMED: UPDATE gt_label for 1,000 patches ===')
try:
    t_start = time.perf_counter()

    cur.execute(
        f"UPDATE {TABLE} SET gt_label = %s, event_ts = NOW() WHERE id = ANY(%s)",
        (NEW_LABEL, patch_ids)
    )
    updated_count = cur.rowcount
    conn.commit()

    t_end = time.perf_counter()
    # -----------------------------------------------------------------------
    # END of timed section
    # -----------------------------------------------------------------------

    elapsed    = t_end - t_start
    throughput = BATCH_SIZE / elapsed

    print(f'\nRows updated  : {updated_count:,} / {BATCH_SIZE}')
    print(f'Elapsed time  : {elapsed:.4f}s')
    print(f'Throughput    : {throughput:,.0f} patches/s')
    print(f'\nRESULT: "{elapsed:.3f}s, {throughput:,.0f} patches/s"')

finally:
    # -----------------------------------------------------------------------
    # RESTORE original gt_label values (NOT timed — guaranteed cleanup)
    # -----------------------------------------------------------------------
    print('\n--- Restoring original gt_label values ---')
    restore_data = [(orig_gt, pid) for pid, orig_gt in original_map.items()]
    psycopg2.extras.execute_batch(
        cur,
        f"UPDATE {TABLE} SET gt_label = %s WHERE id = %s",
        restore_data,
        page_size=500
    )
    conn.commit()
    print(f'Restored gt_label for {len(restore_data):,} rows.')

    # Verify restoration
    cur.execute(
        f'SELECT id, gt_label FROM {TABLE} WHERE id = ANY(%s)',
        (patch_ids,)
    )
    restored = {row[0]: row[1] for row in cur.fetchall()}
    mismatches = [(pid, original_map[pid], restored.get(pid))
                  for pid in original_map if restored.get(pid) != original_map[pid]]
    if mismatches:
        print(f'WARNING: {len(mismatches)} rows not fully restored!')
        for pid, orig, got in mismatches[:5]:
            print(f'  id={pid}: expected gt_label={orig}, got {got}')
    else:
        print('Verification passed — all gt_label values restored correctly.')

conn.close()

Connected to PostgreSQL 15.17 (Debian 15.17-1.pgdg13+1) on x86_64-pc-linux-gnu
ID range: 1 – 800,000,200  |  Table row count: 800,000,200
Sampled 1000 random patch IDs (seed=42)
Saved original gt_label for 1,000 rows

--- Warm-up SELECT (not timed) ---

=== TIMED: UPDATE gt_label for 1,000 patches ===

Rows updated  : 1,000 / 1000
Elapsed time  : 0.0135s
Throughput    : 74,164 patches/s

RESULT: "0.013s, 74,164 patches/s"

--- Restoring original gt_label values ---
Restored gt_label for 1,000 rows.
Verification passed — all gt_label values restored correctly.


In [ ]:
# ---------------------------------------------------------------------------
# 3-trial stability run
# ---------------------------------------------------------------------------
import os
import time
import random
import psycopg2
import psycopg2.extras

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TABLE      = 'v2_patch'
BATCH_SIZE = 1_000
NEW_LABEL  = 99
SEEDS      = [0, 100, 200]

conn = psycopg2.connect(DSN)
conn.autocommit = False
cur  = conn.cursor()

cur.execute(f'SELECT MIN(id), MAX(id) FROM {TABLE};')
min_id, max_id = cur.fetchone()

print('3-trial stability check:')
elapsed_list    = []
throughput_list = []

for seed in SEEDS:
    rng = random.Random(seed)
    patch_ids = rng.sample(range(min_id, max_id + 1), BATCH_SIZE)

    # Save originals
    cur.execute(
        f'SELECT id, gt_label FROM {TABLE} WHERE id = ANY(%s)',
        (patch_ids,)
    )
    original_map = {row[0]: row[1] for row in cur.fetchall()}

    # Warm-up
    cur.execute(
        f'SELECT id FROM {TABLE} WHERE id = ANY(%s)', (patch_ids,)
    )
    _ = cur.fetchall()
    conn.rollback()

    try:
        t_start = time.perf_counter()
        cur.execute(
            f"UPDATE {TABLE} SET gt_label = %s, event_ts = NOW() WHERE id = ANY(%s)",
            (NEW_LABEL, patch_ids)
        )
        conn.commit()
        t_end = time.perf_counter()

        elapsed    = t_end - t_start
        throughput = BATCH_SIZE / elapsed
        elapsed_list.append(elapsed)
        throughput_list.append(throughput)
        print(f'  Trial {SEEDS.index(seed)+1}  seed={seed:<3}: elapsed={elapsed:.4f}s  throughput={throughput:,.0f} patches/s')

    finally:
        # Restore originals
        restore_data = [(orig_gt, pid) for pid, orig_gt in original_map.items()]
        psycopg2.extras.execute_batch(
            cur,
            f"UPDATE {TABLE} SET gt_label = %s WHERE id = %s",
            restore_data,
            page_size=500
        )
        conn.commit()

mean_elapsed    = sum(elapsed_list) / len(elapsed_list)
mean_throughput = sum(throughput_list) / len(throughput_list)
print(f'\nMean elapsed    : {mean_elapsed:.4f}s')
print(f'Mean throughput : {mean_throughput:,.0f} patches/s')

conn.close()

3-trial stability check:
  Trial 1  seed=0  : elapsed=0.0132s  throughput=75,587 patches/s
  Trial 2  seed=100: elapsed=0.0148s  throughput=67,575 patches/s
  Trial 3  seed=200: elapsed=0.0165s  throughput=60,467 patches/s

Mean elapsed    : 0.0149s
Mean throughput : 67,877 patches/s


# Result Summary

## Execution Output (actual run)

```
Connected to PostgreSQL 15.17 (Debian 15.17-1.pgdg13+1) on x86_64-pc-linux-gnu
ID range: 1 – 800,000,200  |  Table row count: 800,000,200
Sampled 1000 random patch IDs (seed=42)
Saved original gt_label for 1,000 rows

--- Warm-up SELECT (not timed) ---

=== TIMED: UPDATE gt_label for 1,000 patches ===

Rows updated  : 1,000 / 1000
Elapsed time  : 0.0135s
Throughput    : 74,164 patches/s

RESULT: "0.013s, 74,164 patches/s"

--- Restoring original gt_label values ---
Restored gt_label for 1,000 rows.
Verification passed — all gt_label values restored correctly.

3-trial stability check:
  Trial 1  seed=0  : elapsed=0.0132s  throughput=75,587 patches/s
  Trial 2  seed=100: elapsed=0.0148s  throughput=67,575 patches/s
  Trial 3  seed=200: elapsed=0.0165s  throughput=60,467 patches/s

Mean elapsed    : 0.0149s
Mean throughput : 67,877 patches/s
```

## Summary

| Metric              | Value |
|---------------------|-------|
| Table size          | 800,000,200 rows (~800M) |
| Updates per trial   | 1,000 |
| Elapsed (seed=42)   | **0.013s** |
| Throughput          | **~74,164 patches/s** |
| Mean (3 trials)     | 0.015s, ~67,877 patches/s |

## Notes

- **Result written to CSV**: `"0.013s, ~74,000 patches/s"`
- Operation: single `UPDATE v2_patch SET gt_label = %s, event_ts = NOW() WHERE id = ANY(%s)` with 1,000 IDs.
- Index: PK B-tree on `id` is used for the `ANY` predicate — O(log N) per ID lookup.
- Table is UNLOGGED (~800M rows) — WAL overhead is minimized.
- IDs were sampled uniformly at random across the full 1–800M range (worst-case scatter access pattern).
- Original `gt_label` values were saved before the timed block and fully restored in a `finally` block.
- Restoration was verified: all 1,000 rows returned to their original labels with no mismatches.
- All three stability trials remain well under 20ms — this operation is extremely fast for any realistic labeling UX.
- Setup/teardown details in `setup_assign_labels_v2_patch.ipynb`.